In [23]:
# init
import importlib, sys

import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import savgol_filter
from matplotlib.markers import MarkerStyle
from matplotlib.lines import Line2D

import superconductivity.api as sc

from superconductivity.api import G0_muS
from superconductivity.api import NDArray64

from IPython import get_ipython

from tqdm import tqdm

Textwidth: float = 4.25279  # in
Textheight: float = 6.85173  # in

_ip = get_ipython()
if _ip is not None:
    _ip.run_line_magic("reload_ext", "autoreload")
    _ip.run_line_magic("autoreload", "2")

    _ip.run_line_magic(
        "config",
        "InlineBackend.print_figure_kwargs = {'bbox_inches': None, 'pad_inches': 0.0}",
    )
    _ip.run_line_magic("config", 'InlineBackend.figure_format = "retina"')  # or "png"
    _ip.run_line_magic("config", "InlineBackend.rc = {'figure.dpi': 300}")

# Single IV fit

In [24]:
GN_G0 = 0.18949935853204078
T_K = 0.21497232546013467
Delta_meV = 0.19479995314857806
gamma_meV = 0.0011282098342488703
sigmaV_mV = 0.010314565303472743

In [25]:
data = np.load("single_iv/eva.npz")
V_mV = data["V_mV"]
I_nA = data["I_nA"]

In [26]:
settings = {
    "GN_G0": (GN_G0, 0.1, 0.3, False),
    "T_K": (T_K, 0.0, 1.2, False),
    "Delta_meV": (Delta_meV, 0.180, 0.210, False),
    "gamma_meV": (gamma_meV, 1e-9, 25e-3, False),
    "sigmaV_mV": (sigmaV_mV, 0.0, 1.0, False),
    "A_mV": (0.0, 0.0, 4.0, True),
    "nu_GHz": (0.0, 1.0, 20.0, True),
}

# fit only the noise-capable model
from bcs_fit import fit_bcs_conv_noise
from compare_bcs_fits import plot_bcs_conv_noise_fit, print_bcs_comparison_table

fit_row, noise_solution = fit_bcs_conv_noise(
    V_mV,
    I_nA,
    settings,
    maxfev=20_000,
)

print_bcs_comparison_table(
    [fit_row],
    columns=(
        "model",
        "ok",
        "rmse_nA",
        "GN_G0",
        "T_K",
        "Delta_meV",
        "gamma_meV",
        "sigmaV_mV",
    ),
)

Ifit_nA = noise_solution["I_fit_nA"]
fit_values = {parameter.name: parameter.value for parameter in noise_solution["params"]}
fit_errors = {parameter.name: parameter.error for parameter in noise_solution["params"]}

fig = plot_bcs_conv_noise_fit(fit_row, noise_solution)
plt.close("all")

print("GN_G0", GN_G0, fit_values["GN_G0"], fit_errors["GN_G0"])
print("T_K", T_K, fit_values["T_K"], fit_errors["T_K"])
print("Delta_meV", Delta_meV, fit_values["Delta_meV"], fit_errors["Delta_meV"])
print("gamma_meV", gamma_meV, fit_values["gamma_meV"], fit_errors["gamma_meV"])
print("sigmaV_mV", sigmaV_mV, fit_values["sigmaV_mV"], fit_errors["sigmaV_mV"])

model           ok    rmse_nA    GN_G0     T_K       Delta_meV  gamma_meV   sigmaV_mV
--------------  ----  ---------  --------  --------  ---------  ----------  ---------
bcs_conv_noise  True  0.0160935  0.189499  0.214961  0.1948     0.00112821  0.0103146
GN_G0 0.18949935853204078 0.18949935749225558 3.544694550177175e-06
T_K 0.21497232546013467 0.21496138361353764 0.012516507016705536
Delta_meV 0.19479995314857806 0.194799914492272 4.507377625873287e-05
gamma_meV 0.0011282098342488703 0.0011282112249668148 3.769542649145138e-06
sigmaV_mV 0.010314565303472743 0.010314562466348246 1.0145884659157968e-05


In [27]:
# saving fit data
np.savez_compressed(
    "single_iv/fit.npz",
    V_mV=V_mV,
    Iexp_nA=I_nA,
    Ifit_nA=Ifit_nA,
    GN_G0=GN_G0,
    T_K=T_K,
    Delta_meV=Delta_meV,
    gamma_meV=gamma_meV,
    sigmaV_mV=sigmaV_mV,
)

# Calibration Amplitude Study @ 18.3 GHz

In [28]:
# load raw data
data = np.load("amp_18.3GHz/eva.npz")

Vbias_mV = data["Vbias_mV"]
Ibias_nA = data["Ibias_nA"]

Aout_mV = data["Aout_mV"]
nu_GHz = data["nu_GHz"]

Iexp_nA = data["Iexp_nA"]
dGexp_G0 = data["dGexp_G0"]
dRexp_R0 = data["dRexp_R0"]

In [29]:
# fit start values
GN_G0: float = 0.18877592218372993
Delta_meV: float = 0.19345000789195935
gamma_meV: float = 0.005066874981090785
eta = 0.002173  # (3)
Tbase_K = 0.0806  # (46) K

# Initial values, bounds, and fixed state.
settings = {
    "GN_G0": (GN_G0, 0.1, 0.3, True),
    "T_K": (Tbase_K, Tbase_K, 1.2, False),
    "Delta_meV": (Delta_meV, 0.180, 0.210, True),
    "gamma_meV": (gamma_meV, 1e-9, 25e-3, True),
    "A_mV": (0.0, 0.0, 4.0, False),
    "nu_GHz": (nu_GHz, 1.0, 20.0, True),
}

In [4]:
# # actual fitting
# from dataclasses import replace

# from superconductivity.optimizers import fit_model
# from superconductivity.optimizers.bcs import get_model_spec

# model = "pat_conv_jax"
# spec = get_model_spec(model)

# parameters = [
#     replace(
#         parameter,
#         guess=settings[parameter.name][0],
#         lower=settings[parameter.name][1],
#         upper=settings[parameter.name][2],
#         fixed=settings[parameter.name][3],
#     )
#     for parameter in spec.parameters
# ]

# n_curves = Iexp_nA.shape[0]

# I_fit_nA = np.full_like(Iexp_nA, np.nan, dtype=float)
# fit_values = {parameter.name: np.full(n_curves, np.nan) for parameter in parameters}
# fit_errors = {parameter.name: np.full(n_curves, np.nan) for parameter in parameters}
# solutions = []

# for index, current_nA in enumerate(tqdm(Iexp_nA)):
#     try:
#         solution = fit_model(
#             Vbias_mV,
#             current_nA,
#             model=model,
#             parameters=parameters,
#             maxfev=2_000,
#         )
#     except (RuntimeError, ValueError) as error:
#         print(f"Curve {index} failed: {error}")
#         solutions.append(None)
#         continue

#     solutions.append(solution)
#     I_fit_nA[index] = solution["I_fit_nA"]

#     for parameter in solution["params"]:
#         fit_values[parameter.name][index] = parameter.value
#         fit_errors[parameter.name][index] = parameter.error

#     # Warm-start the next curve while retaining bounds/fixed state.
#     parameters = [
#         replace(parameter, guess=parameter.value) for parameter in solution["params"]
#     ]

# output = "amp_18.3GHz_fit.npz"

# np.savez_compressed(
#     output,
#     I_fit_nA=I_fit_nA,
#     **{f"value_{name}": data for name, data in fit_values.items()},
#     **{f"error_{name}": data for name, data in fit_errors.items()},
# )

In [5]:
# load fit
data = np.load("amp_18.3GHz/fit_data.npz")

I_fit_nA = data["I_fit_nA"]

fit_values = {
    key.removeprefix("value_"): data[key]
    for key in data.files
    if key.startswith("value_")
}

fit_errors = {
    key.removeprefix("error_"): data[key]
    for key in data.files
    if key.startswith("error_")
}

In [6]:
# binning
from superconductivity.utilities.functions.upsampling import upsample

Acal = np.linspace(0, 5, 101)
Acal_mV = Acal * (nu_GHz * sc.h_pVs)

Afit_mV = fit_values["A_mV"]
Afit = Afit_mV / (nu_GHz * sc.h_pVs)

Afitup = upsample(Afit, N_up=100)
Aoutup_mV = upsample(Aout_mV, N_up=100)
Iexpup_nA = upsample(Iexp_nA, N_up=100, axis=0)
dGexpup_G0 = upsample(dGexp_G0, N_up=100, axis=0)
dRexpup_R0 = upsample(dRexp_R0, N_up=100, axis=0)

valid = np.isfinite(Afitup)

Aoutcal_mV = sc.bin(
    z=Aoutup_mV[valid],
    x=Afitup[valid],
    xbins=Acal,
    axis=0,
)
Ical_nA = sc.bin(
    z=Iexpup_nA[valid, :],
    x=Afitup[valid],
    xbins=Acal,
    axis=0,
)
dGcal_G0 = sc.bin(
    z=dGexpup_G0[valid, :],
    x=Afitup[valid],
    xbins=Acal,
    axis=0,
)
dRcal_R0 = sc.bin(
    z=dRexpup_R0[valid, :],
    x=Afitup[valid],
    xbins=Acal,
    axis=0,
)

In [7]:
# saving calibrated data

Vbias = Vbias_mV / Delta_meV
Ibias = Ibias_nA / (GN_G0 * sc.G0_muS * Delta_meV)
Abias = Acal

Iexp = Ical_nA / (GN_G0 * sc.G0_muS * Delta_meV)
dGexp = dGcal_G0 / (GN_G0)
dRexp = dRcal_R0 * (GN_G0)

np.savez_compressed(
    "amp_18.3GHz/cal.npz",
    Vbias=Vbias,
    Ibias=Ibias,
    Abias=Abias,
    Iexp=Iexp,
    dGexp=dGexp,
    dRexp=dRexp,
    nu_GHz=nu_GHz,
    GN_G0=GN_G0,
    Delta_meV=Delta_meV,
)

In [8]:
# saving fit data
Afit_K = fit_values["A_mV"]
uAfit_mV = fit_errors["A_mV"]
Tfit_K = fit_values["T_K"]
uTfit_K = fit_errors["T_K"]
np.savez_compressed(
    "amp_18.3GHz/fit.npz",
    Aout_mV=Aout_mV,
    Afit_mV=Afit_mV,
    uAfit_mV=uAfit_mV,
    Tfit_K=Tfit_K,
    uTfit_K=uTfit_K,
    nu_GHz=nu_GHz,
    GN_G0=GN_G0,
    Delta_meV=Delta_meV,
)

# Calibration Amplitude Study @ 13.6 GHz

In [6]:
# load raw data
data = np.load("amp_13.6GHz/eva.npz")

Vbias_mV = data["Vbias_mV"]
Ibias_nA = data["Ibias_nA"]
Aout_mV = data["Aout_mV"]

Iexp_nA = data["Iexp_nA"]
dGexp_G0 = data["dGexp_G0"]
dRexp_R0 = data["dRexp_R0"]

nu_GHz = data["nu_GHz"]

In [7]:
# fit start values
GN_G0: float = 0.18877592218372993
Delta_meV: float = 0.19345000789195935
gamma_meV: float = 0.005066874981090785
eta = 0.002173  # (3)
Tbase_K = 0.0806  # (46) K

# Initial values, bounds, and fixed state.
settings = {
    "GN_G0": (GN_G0, 0.1, 0.3, True),
    "T_K": (Tbase_K, 0.0, 1.2, False),
    "Delta_meV": (Delta_meV, 0.180, 0.210, True),
    "gamma_meV": (gamma_meV, 1e-9, 25e-3, True),
    "A_mV": (0.0, 0.0, 4.0, False),
    "nu_GHz": (nu_GHz, 1.0, 20.0, True),
}

In [8]:
# # actual fitting
# from dataclasses import replace

# from superconductivity.optimizers import fit_model
# from superconductivity.optimizers.bcs import get_model_spec

# model = "pat_conv_jax"
# spec = get_model_spec(model)

# parameters = [
#     replace(
#         parameter,
#         guess=settings[parameter.name][0],
#         lower=settings[parameter.name][1],
#         upper=settings[parameter.name][2],
#         fixed=settings[parameter.name][3],
#     )
#     for parameter in spec.parameters
# ]

# n_curves = Iexp_nA.shape[0]

# I_fit_nA = np.full_like(Iexp_nA, np.nan, dtype=float)
# fit_values = {parameter.name: np.full(n_curves, np.nan) for parameter in parameters}
# fit_errors = {parameter.name: np.full(n_curves, np.nan) for parameter in parameters}
# solutions = []

# for index, current_nA in enumerate(tqdm(Iexp_nA)):
#     try:
#         solution = fit_model(
#             Vbias_mV,
#             current_nA,
#             model=model,
#             parameters=parameters,
#             maxfev=2_000,
#         )
#     except (RuntimeError, ValueError) as error:
#         print(f"Curve {index} failed: {error}")
#         solutions.append(None)
#         continue

#     solutions.append(solution)
#     I_fit_nA[index] = solution["I_fit_nA"]

#     for parameter in solution["params"]:
#         fit_values[parameter.name][index] = parameter.value
#         fit_errors[parameter.name][index] = parameter.error

#     # Warm-start the next curve while retaining bounds/fixed state.
#     parameters = [
#         replace(parameter, guess=parameter.value) for parameter in solution["params"]
#     ]

# output = "amp_13.6GHz_fit.npz"

# np.savez_compressed(
#     output,
#     I_fit_nA=I_fit_nA,
#     **{f"value_{name}": data for name, data in fit_values.items()},
#     **{f"error_{name}": data for name, data in fit_errors.items()},
# )

In [9]:
# load fit
data = np.load("amp_13.6GHz/fit_data.npz")

I_fit_nA = data["I_fit_nA"]

fit_values = {
    key.removeprefix("value_"): data[key]
    for key in data.files
    if key.startswith("value_")
}

fit_errors = {
    key.removeprefix("error_"): data[key]
    for key in data.files
    if key.startswith("error_")
}

In [10]:
# binning
from superconductivity.utilities.functions.upsampling import upsample

Acal = np.linspace(0, 30, 101)
Acal_mV = Acal * (nu_GHz * sc.h_pVs)

Afit_mV = fit_values["A_mV"]
Afit = Afit_mV / (nu_GHz * sc.h_pVs)

Afitup = upsample(Afit, N_up=100)
Aoutup_mV = upsample(Aout_mV, N_up=100)
Iexpup_nA = upsample(Iexp_nA, N_up=100, axis=0)
dGexpup_G0 = upsample(dGexp_G0, N_up=100, axis=0)
dRexpup_R0 = upsample(dRexp_R0, N_up=100, axis=0)

valid = np.isfinite(Afitup)

Aoutcal_mV = sc.bin(
    z=Aoutup_mV[valid],
    x=Afitup[valid],
    xbins=Acal,
    axis=0,
)
Ical_nA = sc.bin(
    z=Iexpup_nA[valid, :],
    x=Afitup[valid],
    xbins=Acal,
    axis=0,
)
dGcal_G0 = sc.bin(
    z=dGexpup_G0[valid, :],
    x=Afitup[valid],
    xbins=Acal,
    axis=0,
)
dRcal_R0 = sc.bin(
    z=dRexpup_R0[valid, :],
    x=Afitup[valid],
    xbins=Acal,
    axis=0,
)

In [11]:
# saving calibrated data

Vbias = Vbias_mV / Delta_meV
Ibias = Ibias_nA / (GN_G0 * sc.G0_muS * Delta_meV)
Abias = Acal

Iexp = Ical_nA / (GN_G0 * sc.G0_muS * Delta_meV)
dGexp = dGcal_G0 / (GN_G0)
dRexp = dRcal_R0 * (GN_G0)

np.savez_compressed(
    "amp_13.6GHz/cal.npz",
    Vbias=Vbias,
    Ibias=Ibias,
    Abias=Abias,
    Iexp=Iexp,
    dGexp=dGexp,
    dRexp=dRexp,
    nu_GHz=nu_GHz,
    GN_G0=GN_G0,
    Delta_meV=Delta_meV,
)

In [12]:
# saving fit data
Afit_K = fit_values["A_mV"]
uAfit_mV = fit_errors["A_mV"]
Tfit_K = fit_values["T_K"]
uTfit_K = fit_errors["T_K"]
np.savez_compressed(
    "amp_13.6GHz/fit.npz",
    Aout_mV=Aout_mV,
    Afit_mV=Afit_mV,
    uAfit_mV=uAfit_mV,
    Tfit_K=Tfit_K,
    uTfit_K=uTfit_K,
    nu_GHz=nu_GHz,
    GN_G0=GN_G0,
    Delta_meV=Delta_meV,
)

# Calibration Temperature Study

In [2]:
# load raw data
data = np.load("temperature/eva.npz")

Vbias_mV = data["Vbias_mV"]
Ibias_nA = data["Ibias_nA"]
Tbath_K = data["Tbath_K"]

Iexp_nA = data["Iexp_nA"]
dGexp_G0 = data["dGexp_G0"]
dRexp_R0 = data["dRexp_R0"]

In [3]:
# fit start values
GN_G0: float = 0.18877592218372993
Delta_meV: float = 0.19345000789195935
gamma_meV: float = 0.005066874981090785
eta = 0.002173  # (3)
Tbase_K = 0.0806  # (46) K

# Initial values, bounds, and fixed state.
settings = {
    "GN_G0": (GN_G0, 0.1, 0.3, True),
    "T_K": (0.2, 0.0, 1.5, False),
    "Delta_meV": (Delta_meV, 0.180, 0.210, True),
    "gamma_meV": (gamma_meV, 1e-9, 25e-3, True),
    "A_mV": (0.0, 0.0, 4.0, True),
    "nu_GHz": (0.0, 1.0, 20.0, True),
}

In [4]:
# # actual fitting
# from dataclasses import replace

# from superconductivity.optimizers import fit_model
# from superconductivity.optimizers.bcs import get_model_spec

# model = "pat_conv_jax"
# spec = get_model_spec(model)

# parameters = [
#     replace(
#         parameter,
#         guess=settings[parameter.name][0],
#         lower=settings[parameter.name][1],
#         upper=settings[parameter.name][2],
#         fixed=settings[parameter.name][3],
#     )
#     for parameter in spec.parameters
# ]

# n_curves = Iexp_nA.shape[0]

# I_fit_nA = np.full_like(Iexp_nA, np.nan, dtype=float)
# fit_values = {parameter.name: np.full(n_curves, np.nan) for parameter in parameters}
# fit_errors = {parameter.name: np.full(n_curves, np.nan) for parameter in parameters}
# solutions = []

# for index, current_nA in enumerate(tqdm(Iexp_nA)):
#     try:
#         solution = fit_model(
#             Vbias_mV,
#             current_nA,
#             model=model,
#             parameters=parameters,
#             maxfev=2_000,
#         )
#     except (RuntimeError, ValueError) as error:
#         print(f"Curve {index} failed: {error}")
#         solutions.append(None)
#         continue

#     solutions.append(solution)
#     I_fit_nA[index] = solution["I_fit_nA"]

#     for parameter in solution["params"]:
#         fit_values[parameter.name][index] = parameter.value
#         fit_errors[parameter.name][index] = parameter.error

#     # Warm-start the next curve while retaining bounds/fixed state.
#     parameters = [
#         replace(parameter, guess=parameter.value) for parameter in solution["params"]
#     ]

# output = "temperature/fit_data.npz"

# np.savez_compressed(
#     output,
#     I_fit_nA=I_fit_nA,
#     **{f"value_{name}": data for name, data in fit_values.items()},
#     **{f"error_{name}": data for name, data in fit_errors.items()},
# )

In [27]:
# load fit
data = np.load("temperature/fit_data.npz")

fit_values = {
    key.removeprefix("value_"): data[key]
    for key in data.files
    if key.startswith("value_")
}

fit_errors = {
    key.removeprefix("error_"): data[key]
    for key in data.files
    if key.startswith("error_")
}

In [28]:
# binning
from superconductivity.utilities.functions.upsampling import upsample

Tc_K = 1.26
Tcal = np.linspace(0.5, 1.0, 101)
Tcal_K = Tcal * Tc_K

Tfit_K = fit_values["T_K"]
uTfit_K = fit_errors["T_K"]

Tfitup_K = upsample(Tfit_K, N_up=100)
Tbathup_K = upsample(Tbath_K, N_up=100)
Iexpup_nA = upsample(Iexp_nA, N_up=100, axis=0)
dGexpup_G0 = upsample(dGexp_G0, N_up=100, axis=0)
dRexpup_R0 = upsample(dRexp_R0, N_up=100, axis=0)

valid = np.isfinite(Tfitup_K)

Tbathcal_mV = sc.bin(
    z=Tbathup_K[valid],
    x=Tfitup_K[valid],
    xbins=Tcal_K,
    axis=0,
)
Ical_nA = sc.bin(
    z=Iexpup_nA[valid, :],
    x=Tfitup_K[valid],
    xbins=Tcal_K,
    axis=0,
)
dGcal_G0 = sc.bin(
    z=dGexpup_G0[valid, :],
    x=Tfitup_K[valid],
    xbins=Tcal_K,
    axis=0,
)
dRcal_R0 = sc.bin(
    z=dRexpup_R0[valid, :],
    x=Tfitup_K[valid],
    xbins=Tcal_K,
    axis=0,
)

In [29]:
# saving calibrated data

Vbias = Vbias_mV / Delta_meV
Ibias = Ibias_nA / (GN_G0 * sc.G0_muS * Delta_meV)
Iexp = Ical_nA / (GN_G0 * sc.G0_muS * Delta_meV)
dGexp = dGcal_G0 / (GN_G0)
dRexp = dRcal_R0 * (GN_G0)

np.savez_compressed(
    "temperature/cal.npz",
    Vbias=Vbias,
    Ibias=Ibias,
    Tcal=Tcal,
    Iexp=Iexp,
    dGexp=dGexp,
    dRexp=dRexp,
    GN_G0=GN_G0,
    Delta_meV=Delta_meV,
)

In [30]:
from scipy.optimize import curve_fit


def calibration_T(T: NDArray64, T_base: float, T_off: float, alpha: float):
    T = T_off + alpha * T
    return np.where(T <= T_base, T_base, T)


mask0 = Tbath_K <= 1.3

popt, pcov = curve_fit(
    f=calibration_T,
    xdata=np.array(Tbath_K[mask0], dtype="float64"),
    ydata=np.array(Tfit_K[mask0], dtype="float64"),
    sigma=np.array(uTfit_K[mask0], dtype="float64"),
)
perr = np.sqrt(np.diag(pcov))

In [35]:
mask1 = Tbath_K > 1.42
mask1 = np.logical_not(mask0)
Tcritical_K = np.median(Tfit_K[mask1])
uTcritical_K = np.std(Tfit_K[mask1])
print("Tcritical_K =", Tcritical_K, uTcritical_K)

Tbase_K, uTbase_K = popt[0], perr[0]
print("Tbase_K =", Tbase_K, uTbase_K)

Toff_K, uToff_K = popt[1], perr[1]
print("Toff_K =", Toff_K, uToff_K)

alphaT, ualphaT = popt[2], perr[2]
print("alphaT =", alphaT, ualphaT)

Tcritical_K = 1.2611020935028985 0.016131157467138363
Tbase_K = 0.2893960917964936 0.005888717423464449
Toff_K = -0.06903118678057414 0.0019410011931929882
alphaT = 0.9808868229283363 0.001700655841730937


In [36]:
# saving fit data
Tfit_K = fit_values["T_K"]
uTfit_mV = fit_errors["T_K"]
np.savez_compressed(
    "temperature/fit.npz",
    Tbath_K=Tbath_K,
    Tfit_K=Tfit_K,
    uTfit_K=uTfit_K,
    Tcritical_K=Tcritical_K,
    uTcritical_K=uTcritical_K,
    Tbase_K=Tbase_K,
    uTbase_K=uTbase_K,
    Toff_K=Toff_K,
    uToff_K=uToff_K,
    alphaT=alphaT,
    ualphaT=ualphaT,
    mask0=mask0,
    mask1=mask1,
    GN_G0=GN_G0,
    Delta_meV=Delta_meV,
)